# 07 — Báo cáo tổng hợp

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/vinumqa-numerical-reasoning/blob/main/notebooks/07_final_report.ipynb)

**Không cần GPU.** Runtime → CPU. Khoảng 1 phút.

## Mục đích

Gộp mọi thứ đã chạy — 5 nấc của chuỗi (notebook 01–05) và các ô của ma trận tổ hợp
(notebook 06) — thành một báo cáo duy nhất:

1. **Bảng thang bậc** — mỗi nấc và phần tăng thêm so với nấc ngay trước.
2. **Kiểm định McNemar** cho từng bước leo thang.
3. **Phân tích theo độ phức tạp và theo nguồn dữ liệu** — kỹ thuật nào giúp loại bài nào.
4. **Chuyển dịch kiểu lỗi** qua các nấc.
5. **Đối chiếu mốc tham chiếu** — `PA_loose` so với con số tham chiếu.
6. Biểu đồ + hai bảng CSV để dán thẳng vào bài.

Notebook này **không chạy model**, chỉ đọc lại kết quả đã lưu, nên chạy lại bao nhiêu lần
cũng được và nấc nào chưa chạy thì tự bỏ qua.

## §1. Môi trường

In [ ]:
%%capture
!pip install -q pandas matplotlib

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CẤU HÌNH — CHỈ SỬA MỘT DÒNG, MỘT LẦN, Ở NOTEBOOK 00                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Code + dữ liệu lấy thẳng từ GitHub: cell này tự clone lần đầu và tự cập nhật
# những lần sau, nên sửa code dưới máy chỉ cần `git push` là Colab có bản mới.
# Riêng KẾT QUẢ ghi lên Drive để không mất khi Colab ngắt session.
GITHUB_REPO = "https://github.com/ThanhDatVN/vinumqa-numerical-reasoning"
OUTPUT_DIR  = "/content/drive/MyDrive/vinumqa_runs"

# Hai dòng dưới để trống là được — chỉ điền khi muốn tự quyết:
#   REPO_DIR  chỗ đã có sẵn code, điền vào thì bỏ qua bước clone
#   DATA_DIR  chỗ để dữ liệu, nếu tách khỏi code
REPO_DIR = ""
DATA_DIR = ""
# ──────────────────────────────────────────────────────────────────────────────

import os, sys, json, time, csv, gc, random, glob, shutil, subprocess
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np

_PLACEHOLDER = "TEN-TAI-KHOAN"


def _is_repo(p):
    """Thư mục p có phải bản sao của dự án không."""
    return bool(p) and os.path.isdir(os.path.join(p, "vinumqa"))


# Điền sẵn REPO_DIR = đã tự lo chỗ để code, cell này không đụng gì tới git.
_pinned = bool(str(REPO_DIR).strip())

ON_COLAB  = "COLAB_" in "".join(os.environ.keys())
_MEMO = ("/content/drive/MyDrive/.vinumqa_paths.json" if ON_COLAB
         else os.path.join(os.path.expanduser("~"), ".vinumqa_paths.json"))

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
else:                                  # chạy dưới máy: repo là thư mục đang đứng, hoặc cha nó
    _here = os.path.abspath(os.getcwd())
    for _c in (_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))):
        if _is_repo(_c):
            REPO_DIR, OUTPUT_DIR, _pinned = _c, os.path.join(_c, "runs"), True; break

# ─── Ghi nhớ cấu hình: GITHUB_REPO chỉ phải điền một lần, ở notebook 00 ───
_saved = {}
if os.path.exists(_MEMO):
    try:
        _saved = json.load(open(_MEMO, encoding="utf-8"))
    except Exception:
        _saved = {}
if _PLACEHOLDER in GITHUB_REPO and _saved.get("GITHUB_REPO"):
    GITHUB_REPO = _saved["GITHUB_REPO"]
    print("[CẤU HÌNH] dùng GITHUB_REPO đã ghi nhớ từ lần chạy trước")
DATA_DIR = DATA_DIR or _saved.get("DATA_DIR", "")

# ─── Lấy code + dữ liệu về ───
if _pinned:                                      # code đã có sẵn, không clone
    if not _is_repo(REPO_DIR):
        raise FileNotFoundError(
            f"Không thấy package tại {REPO_DIR}/vinumqa.\n"
            f"REPO_DIR phải trỏ tới thư mục chứa vinumqa/, data/, notebooks/ — "
            f"hoặc để trống REPO_DIR để tự clone từ GITHUB_REPO.")
    print(f"[CODE] {REPO_DIR} (chỉ định sẵn)")
else:
    if _PLACEHOLDER in GITHUB_REPO:
        raise ValueError(
            "Chưa điền GITHUB_REPO ở ĐẦU CELL NÀY.\n\n"
            "Sửa thành URL repo của bạn, ví dụ:\n"
            "    GITHUB_REPO = \"https://github.com/ten-cua-ban/vinumqa-ladder\"\n\n"
            "Chỉ cần sửa MỘT LẦN ở notebook 00 — bảy notebook sau tự đọc lại.")
    _url  = GITHUB_REPO.strip().rstrip("/")
    _url  = _url if _url.endswith(".git") else _url + ".git"
    REPO_DIR = os.path.join("/content" if ON_COLAB else os.getcwd(),
                            os.path.basename(_url)[:-len(".git")])
    if _is_repo(REPO_DIR):        # còn lại sau khi restart runtime → lấy bản mới nhất
        _g = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"],
                            capture_output=True, text=True)
        print("[CODE] " + REPO_DIR + " — " +
              ("đã cập nhật bản mới nhất" if _g.returncode == 0 else "giữ bản đang có"))
    else:
        if os.path.exists(REPO_DIR) and os.listdir(REPO_DIR) \
                and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
            raise RuntimeError(
                f"{REPO_DIR} đã tồn tại và không phải bản clone của dự án.\n"
                f"Xoá nó, hoặc điền REPO_DIR ở đầu cell này cho trỏ đúng chỗ có code.")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        print(f"[CODE] đang clone {_url} … (~25 MB, khoảng 15 giây)")
        _g = subprocess.run(["git", "clone", "--depth", "1", _url, REPO_DIR],
                            capture_output=True, text=True)
        if _g.returncode or not _is_repo(REPO_DIR):
            raise RuntimeError(
                "git clone thất bại:\n" + (_g.stderr or "")[-800:] + "\n\n"
                "Kiểm tra lại URL. Nếu repo để private thì dùng dạng có token:\n"
                "    https://<token>@github.com/<tài-khoản>/<repo>")
        print(f"[CODE] → {REPO_DIR}")

sys.path.insert(0, REPO_DIR)

try:                                   # ghi nhớ cho các notebook sau
    json.dump({"GITHUB_REPO": GITHUB_REPO, "REPO_DIR": REPO_DIR,
               "OUTPUT_DIR": OUTPUT_DIR, "DATA_DIR": DATA_DIR},
              open(_MEMO, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
except Exception:
    pass

from vinumqa import data, dsl, io_utils, pipeline, stats

# ─── Bố cục thư mục làm việc ───
DATA_DIR    = DATA_DIR or os.path.join(REPO_DIR, "data")
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Không thấy dữ liệu tại {DATA_DIR} (cần train.json / valid.json / test.json).\n"
        f"Dữ liệu nằm trong repo, nên thường là do repo thiếu thư mục data/ "
        f"— kiểm tra đã push data/ lên GitHub chưa, hoặc điền DATA_DIR ở đầu cell này.")
RESULT_DIR  = os.path.join(OUTPUT_DIR, "stages")      # kết quả từng nấc (dùng chung)
LOG_DIR     = os.path.join(OUTPUT_DIR, "logs")        # output thô của model
ARTIFACT_DIR= os.path.join(OUTPUT_DIR, "artifacts")   # playbook, adapter, biểu đồ
for _d in (OUTPUT_DIR, RESULT_DIR, LOG_DIR, ARTIFACT_DIR):
    os.makedirs(_d, exist_ok=True)

STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

splits = data.load_all(DATA_DIR)
train_all = [s for s in splits["train"] if data.has_gold(s)]
valid_all = [s for s in splits["valid"] if data.has_gold(s)]
test_all  = splits["test"]


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ĐỌC / GHI KẾT QUẢ CÁC NẤC                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Mọi notebook ghi kết quả vào RESULT_DIR theo cùng một quy ước, nên notebook
# sau đọc lại được của notebook trước mà không phải chỉnh đường dẫn.

# Thang prompt LỒNG NHAU: basic ⊂ no_fewshot ⊂ engineered — mỗi nấc thêm đúng một khối
# cắt ra từ prompt hoàn chỉnh, phần chung giống nhau từng ký tự.
LADDER = [
    ("01_basic",           "Nấc 1 — prompt cơ bản (danh sách phép toán + yêu cầu)"),
    ("02_prompt_eng",      "Nấc 2 — prompt hoàn chỉnh (+ hướng dẫn từ khoá + few-shot)"),
    ("03_sft",             "Nấc 3 — + SFT Qwen3-8B"),
    ("04_selfeval_base",   "Nấc 4 — + self-eval (model gốc)"),
    ("05_ace_base",        "Nấc 5 — + ACE (model gốc)"),
    ("05c_ace_basic_base", "Nấc 5c — ACE trên prompt cơ bản"),
    ("05_ace_random_base", "Đối chứng — bullet ngẫu nhiên"),
    ("05c_ace_basic_random_base", "Đối chứng — bullet ngẫu nhiên, prompt cơ bản"),
    ("06_comb_E_A",        "Tổ hợp — prompt + ACE (không self-eval)"),
    ("08_tu_nhat_quan",    "Mới — self-consistency K mẫu (ví dụ cố định)"),
    ("09_vidu_dong",       "Mới — self-consistency + ví dụ truy hồi"),
    ("04_selfeval_base_moi", "Mục tiêu — self-eval + K mẫu + ví dụ truy hồi"),
    ("10_bo_chon",         "Mới — model tự chấm giữa các ứng viên"),
]
LADDER_LABEL = dict(LADDER)

# Những biến phải đi kèm kết quả thì mới truy lại được về sau.
_CFG_KEYS = ("MODEL_NAME", "MODEL_TAG", "TEMPERATURE", "MAX_TOKENS", "REPETITION_PENALTY",
             "MAX_SEQ_LENGTH", "BATCH_SIZE", "GPU_MEM_UTIL", "MAX_NUM_SEQS", "RANDOM_SEED")


def run_env():
    """Môi trường THẬT lúc chạy: commit, GPU, phiên bản thư viện.

    Chỉ đọc thư viện đã nạp (``sys.modules``) chứ không import thêm — vừa nhanh,
    vừa báo đúng những gì thật sự được dùng.
    """
    env = {"python": sys.version.split()[0]}
    try:                                   # bản code nào sinh ra kết quả này
        def _g(*a):
            return subprocess.run(["git", "-C", REPO_DIR, *a],
                                  capture_output=True, text=True).stdout.strip()
        env["commit"] = _g("rev-parse", "--short", "HEAD")
        env["branch"] = _g("rev-parse", "--abbrev-ref", "HEAD")
        env["dirty"] = bool(_g("status", "--porcelain"))
    except Exception:
        pass
    _torch = sys.modules.get("torch")
    if _torch is not None:
        env["torch"] = getattr(_torch, "__version__", "?")
        try:
            if _torch.cuda.is_available():
                _p = _torch.cuda.get_device_properties(0)
                env["gpu"] = _p.name
                env["vram_gb"] = round(_p.total_memory / 1024**3, 1)
                env["cc"] = f"{_p.major}.{_p.minor}"
            else:
                env["gpu"] = "CPU"
        except Exception:
            pass
    else:
        env["gpu"] = "CPU (không nạp torch)"
    for _lib in ("transformers", "trl", "peft", "vllm", "unsloth",
                 "sentence_transformers", "numpy"):
        _m = sys.modules.get(_lib)
        if _m is not None and hasattr(_m, "__version__"):
            env[_lib] = _m.__version__
    return env


def stage_path(stage, kind="jsonl"):
    """Đường dẫn chuẩn của một nấc. kind ∈ {jsonl, meta}."""
    return os.path.join(RESULT_DIR, {
        "jsonl": f"{stage}.jsonl",
        "meta":  f"{stage}_meta.json"}[kind])


# Cấu hình CHUẨN của cả thang bậc — đo trên A100 40GB.
# MAX_SEQ_LENGTH đổi theo GPU (A100 15000 / L4 13500 / T4 8192), mà đổi GPU là đổi
# thành phần lô, đổi kernel, đổi luôn token được lấy mẫu ở temperature 0.1. Hai lần
# chạy khác max_seq KHÔNG so thẳng được, nên phải ghi sang tên nấc khác.
MAX_TOKENS_CHUAN, MAX_SEQ_CHUAN = 4096, 17000



def save_stage(stage, rows, metrics, extra=None, quiet=False):
    """Ghi kết quả một nấc: jsonl + meta, kèm một file output thô.

    Chạy với ``MAX_TOKENS`` khác mức chuẩn thì tự ghi sang tên nấc khác. Đổi trần sinh
    là đổi cấu hình, kết quả không so thẳng với thang bậc được — mà nếu cứ ghi đè lên
    tên cũ thì mất luôn bản chuẩn, không lấy lại được nếu không chạy lại GPU.
    """
    _hau_to = ""
    _mt, _ms = globals().get("MAX_TOKENS"), globals().get("MAX_SEQ_LENGTH")
    if _mt and _mt != MAX_TOKENS_CHUAN:
        _hau_to += f"_tok{_mt}"
    if _ms and _ms != MAX_SEQ_CHUAN:
        _hau_to += f"_seq{_ms}"
    if _hau_to and not stage.endswith(_hau_to):
        stage = f"{stage}{_hau_to}"
        if not quiet:
            print(f"[GHI] ⚠ cấu hình khác chuẩn (max_tokens={_mt}, max_seq={_ms}, "
                  f"GPU={globals().get('_GPU', '?')}) → ghi sang nấc '{stage}'.")
            print( "       Kết quả khác GPU/khác trần KHÔNG so thẳng với thang bậc chuẩn.")
    io_utils.save_full_jsonl(rows, stage_path(stage, "jsonl"))
    io_utils.save_raw_jsonl(rows, os.path.join(LOG_DIR, f"{stage}_raw_{STAMP}.jsonl"))
    meta = {"stage": stage, "label": LADDER_LABEL.get(stage, stage),
            "stamp": STAMP, "n": len(rows), "metrics": metrics,
            "model": globals().get("MODEL_NAME"),
            "temperature": globals().get("TEMPERATURE"),
            "max_tokens": globals().get("MAX_TOKENS"),
            "max_seq_length": globals().get("MAX_SEQ_LENGTH"),
            "ctx_truncated": bool(getattr(globals().get("prompt_kit", None),
                                          "max_ctx_chars", None)),
            "enable_thinking": getattr(globals().get("prompt_kit", None),
                                       "enable_thinking", "?"),
            "ty_le_bi_cat_token": (round(ty_le_bi_cat(), 4)
                                   if "ty_le_bi_cat" in globals() else None),
            "bi_cat_theo_buoc": (bi_cat_theo_buoc()
                                if "bi_cat_theo_buoc" in globals() else None),
            "nap_an_toan": bool(globals().get("NAP_AN_TOAN", False)),
            "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
            "env": run_env(),
            **(extra or {})}
    with open(stage_path(stage, "meta"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=1, default=str)
    if not quiet:
        print(f"[GHI] nấc '{stage}':")
        print(f"      {stage_path(stage, 'jsonl')}   ← notebook sau đọc file này")
        print(f"      {stage_path(stage, 'meta')}")
    return stage                      # tên THẬT, có thể khác tên truyền vào


def load_stage(stage, quiet=False):
    """Đọc lại kết quả một nấc, đã sắp đúng thứ tự test_all. None nếu chưa có."""
    p = stage_path(stage, "jsonl")
    if not os.path.exists(p):
        if not quiet:
            print(f"[ĐỌC] ⚠ chưa có '{stage}' — chạy notebook tương ứng trước.")
        return None
    rows = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    order = {s["id"]: i for i, s in enumerate(test_all)}
    rows.sort(key=lambda r: order.get(r["id"], 10**9))   # ghép cặp phải cùng thứ tự
    if not quiet:
        print(f"[ĐỌC] '{stage}': {len(rows)} mẫu")
    return rows


def stage_status():
    """Bảng trạng thái: nấc nào đã chạy, kết quả bao nhiêu."""
    print(f"\n{'─'*76}")
    print(f"  TIẾN ĐỘ — {RESULT_DIR}")
    print(f"{'─'*76}")
    print(f"  {'nấc':<22}{'':<4}{'n':>5}{'EA':>9}{'PA_strict':>11}{'chạy lúc':>16}")
    done = 0
    for stage, label in LADDER:
        mp = stage_path(stage, "meta")
        if not os.path.exists(mp):
            print(f"  {stage:<22}{'⊘':<4}{'—':>5}{'—':>9}{'—':>11}{'chưa chạy':>16}")
            continue
        m = json.load(open(mp, encoding="utf-8"))
        mt = m.get("metrics", {})
        done += 1
        print(f"  {stage:<22}{'✓':<4}{m.get('n','?'):>5}{mt.get('EA',0):>9.4f}"
              f"{mt.get('PA_strict',0):>11.4f}{m.get('stamp','?'):>16}")
    print(f"{'─'*76}\n  {done}/{len(LADDER)} nấc đã có kết quả")
    return done


_env = "Colab" if ON_COLAB else "máy cá nhân"
print(f"[MÔI TRƯỜNG] {_env} | vinumqa v{__import__('vinumqa').__version__}")
print(f"[REPO]  {REPO_DIR}")
print(f"[RA]    {OUTPUT_DIR}")
print(f"          ├─ stages/     kết quả từng nấc (jsonl + meta)")
print(f"          ├─ logs/       output thô của model")
print(f"          └─ artifacts/  playbook, adapter, biểu đồ")
print(f"[DỮ LIỆU] {DATA_DIR}")
print(f"          train={len(train_all)} valid={len(valid_all)} test={len(test_all)}")
stage_status()

## §2. Nạp kết quả các nấc

Notebook nào chưa chạy thì bỏ qua, phần còn lại vẫn báo cáo được.

In [ ]:
R, M, META = {}, {}, {}
for stage, label in LADDER:
    rows = load_stage(stage, quiet=True)
    if rows is None:
        print(f"  ⊘ {stage:<22} chưa có kết quả")
        continue
    # Nấc chạy trước khi có trường `ly_do_khong_chay` vẫn phân tích được: điền bù từ
    # bảng của mẫu. Không tốn GPU, chỉ cần file jsonl đã có sẵn.
    _bu = pipeline.bo_sung_ly_do(rows, test_all)
    R[stage] = rows
    M[stage] = pipeline.summarize(rows, label)
    _mp = stage_path(stage, "meta")
    META[stage] = json.load(open(_mp, encoding="utf-8")) if os.path.exists(_mp) else {}
    print(f"  ✓ {stage:<22} {len(rows)} mẫu | EA={M[stage]['EA']:.4f}")

assert R, "Chưa có nấc nào được chạy."

## §3. Bảng thang bậc

In [ ]:
NHANH_RE = ("05_ace_random", "05c_ace_basic")
# Nấc hậu tố `_moi` là hai ô MỤC TIÊU chạy cấu hình sinh KHÁC (K mẫu, temp cao).
# Chúng không phải bậc của thang, nên không xếp vào chuỗi so liên tiếp — nếu xếp thì
# hiệu số giữa hai bậc kề nhau lẫn cả phần đổi cấu hình sinh.
_main = [s for s, _ in LADDER if s in R and not s.startswith(NHANH_RE)
         and not s.endswith("_moi")]

# ═══ KIỂM TRA CÔNG BẰNG ═══
# So sánh giữa các nấc chỉ có nghĩa khi mọi thứ NGOÀI kỹ thuật đang đo đều giống nhau.
# Từng có lần hai nấc chạy trên hai GPU khác nhau (L4 và A100) — chênh lệch gồm cả ảnh
# hưởng phần cứng, không quy về prompt được.
print(f"\n{'═'*104}\n  KIỂM TRA CÔNG BẰNG — mọi nấc có cùng điều kiện không\n{'═'*104}")
_ck = {}
for s in [x for x, _ in LADDER if x in R]:
    _m = META.get(s, {})
    _e, _c = _m.get("env", {}), _m.get("config", {})
    _ck[s] = (_e.get("gpu", "?"), _c.get("MAX_SEQ_LENGTH", _m.get("max_seq_length")),
              _c.get("MAX_TOKENS", _m.get("max_tokens")),
              _m.get("enable_thinking", "?"), bool(_m.get("ctx_truncated")),
              _m.get("prompt_level", "?"),
              _m.get("temperature_moi", _c.get("TEMPERATURE", _m.get("temperature"))),
              _m.get("so_mau", 1))
print(f"  {'nấc':<24}{'GPU':<20}{'max_tok':>8}{'suy nghĩ':>9}"
      f"{'cắt ng.cảnh':>12}{'nền prompt':>14}{'temp':>7}{'K mẫu':>7}")
for s, v in _ck.items():
    print(f"  {s:<24}{str(v[0])[:19]:<20}{str(v[2]):>8}"
          f"{('bật' if v[3] is None else str(v[3])):>9}"
          f"{('CÓ' if v[4] else 'không'):>12}{str(v[5]):>14}"
          f"{str(v[6]):>7}{str(v[7]):>7}")
print("  (temp/K chỉ để ĐỌC. Nấc 08/09 và ô *_moi cố ý dùng temp cao + K mẫu;")
print("   đó là thiết kế, không phải lệch điều kiện.)")

# Nền prompt chỉ được phép khác ở nấc 1 (basic) và nấc 5c (ACE trên basic).
_MIEN_NEN = ("01_basic", "05c_ace_basic_base")
_nen = {v[5] for s, v in _ck.items() if s not in _MIEN_NEN}

_ten = {0: "GPU", 1: "max_seq", 2: "max_tokens", 3: "chế độ suy nghĩ", 4: "cắt ngữ cảnh"}
_loi = [_ten[i] for i in range(5) if len({v[i] for v in _ck.values()}) > 1]
if len(_nen) > 1:
    _loi.append("nền prompt")
    print("")
    print(f"  ⛔ Các nấc 02/03/04/05/06 KHÔNG cùng nền prompt: {sorted(_nen)}")
    print("     Chỉ nấc 1 và nấc 5c được phép dùng prompt 'basic'. Nấc khác lệch nền thì")
    print("     hiệu số giữa các ô gồm cả phần đổi prompt, không quy cho kỹ thuật được.")

# Thiếu metadata thì MỌI nấc đều bằng nhau ở giá trị rỗng, phép so trên báo "đồng nhất"
# và in ra một lời trấn an SAI. Phải bắt riêng trường hợp này.
_thieu = [s for s in _ck if not META.get(s)]
if _thieu:
    print(f"\n  ⚠ KHÔNG KIỂM ĐƯỢC {len(_thieu)}/{len(_ck)} nấc — thiếu '<nấc>_meta.json': "
          f"{', '.join(_thieu)}")
    print("     Không có metadata thì không kết luận được gì về tính công bằng.")
if _loi:
    print(f"\n  ⛔ KHÔNG ĐỒNG NHẤT ở: {', '.join(_loi)}")
    print("     Chênh lệch giữa các nấc gồm CẢ ảnh hưởng của những thứ này, không quy")
    print("     riêng cho kỹ thuật được. Muốn kết luận chắc thì chạy lại cùng điều kiện.")
elif not _thieu:
    print(f"\n  ✅ Mọi nấc cùng GPU, cùng trần token, cùng chế độ suy nghĩ, không cắt ngữ cảnh.")

_at = [s for s in _ck if META.get(s, {}).get("nap_an_toan")]
if _at:
    print(f"\n  ⚠ Nạp model ở CHẾ ĐỘ AN TOÀN ở: {', '.join(_at)}")
    print("     Nấc đó chạy với util/max_num_seqs thấp hơn (vLLM không dựng nổi engine).")
    print("     Chỉ đổi tốc độ, không đổi token sinh ra vì mỗi request có seed riêng —")
    print("     nhưng ghi lại để còn truy được nếu con số trông lạ.")

_cat = {s: META.get(s, {}).get("ty_le_bi_cat_token") for s in _ck}
_cat = {s: v for s, v in _cat.items() if v is not None}
if _cat:
    print(f"\n  Tỉ lệ lượt sinh bị cắt vì chạm trần token (tách theo bước sinh):")
    print(f"    {'nấc':<24}{'chung':>8}{'bước 1':>9}{'bước 2':>9}")
    for s, v in _cat.items():
        _b = META.get(s, {}).get("bi_cat_theo_buoc") or {}
        _f = lambda k: (f"{_b[k]['ty_le']:.2%}" if k in _b else "—")
        print(f"    {s:<24}{v:>8.2%}{_f('step1'):>9}{_f('step2'):>9}"
              + ("   ⚠" if v > 0.01 else ""))
    # Bước 2 bị cắt nhiều hơn bước 1 = phương pháp mất cơ hội sửa, không phải sửa sai.
    _loang = [s for s in _cat
              if (META.get(s, {}).get("bi_cat_theo_buoc") or {}).get("step2", {}).get("ty_le", 0)
              > (META.get(s, {}).get("bi_cat_theo_buoc") or {}).get("step1", {}).get("ty_le", 0) + 0.02]
    if _loang:
        print(f"\n    ⚠ Bước 2 bị cắt nhiều hơn bước 1 ở: {', '.join(_loang)}")
        print("      Prompt bước 2 chứa cả lời giải bước 1 nên dài hơn. Ở những nấc này")
        print("      hiệu số đo được là CẬN DƯỚI — phương pháp mất bớt cơ hội sửa.")

print(f"\n{'═'*104}\n  KẾT QUẢ THEO NẤC — Qwen3-8B, ViNumQA test ({len(test_all)} mẫu)\n{'═'*104}")
print(f"{'nấc':<34}{'EA':>9}{'PA_strict':>11}{'PA_loose':>11}{'no_prog':>10}"
      f"{'exec_fail':>11}{'phút':>8}")
print("-" * 104)
for s in _main:
    m, meta = M[s], META.get(s, {})
    print(f"{m['label']:<34}{m['EA']:>9.4f}{m['PA_strict']:>11.4f}{m['PA_loose']:>11.4f}"
          f"{m['no_program']:>10.4f}{m['exec_none']:>11.4f}"
          f"{meta.get('metrics', {}).get('minutes', 0):>8.1f}")

# ═══ PHẦN TĂNG THÊM CỦA TỪNG KỸ THUẬT ═══
# Thang bậc là CÂY, không phải một dây. So với DÒNG LIỀN TRÊN trong bảng là sai ở mọi
# chỗ rẽ nhánh: "03_sft → 04_selfeval_base" không đo việc thêm bất cứ kỹ thuật nào, vì
# nấc 4 KHÔNG chạy trên adapter. Ba trong bảy dòng của bản cũ sai đúng kiểu đó.
# Mỗi nấc ở đây so với đúng nấc nó xây lên.
CHA = {
    "02_prompt_eng":        ("01_basic",         "prompt engineering"),
    "03_sft":               ("02_prompt_eng",    "SFT trên dữ liệu tự sinh"),
    "04_selfeval_base":     ("02_prompt_eng",    "self-eval (bước 2 tự soát)"),
    "05_ace_base":          ("04_selfeval_base", "ACE"),
    "06_comb_E_A":          ("02_prompt_eng",    "ACE, không self-eval"),
    "08_tu_nhat_quan":      ("02_prompt_eng",    "self-consistency K=5"),
    "09_vidu_dong":         ("08_tu_nhat_quan",  "ví dụ truy hồi kNN"),
    "04_selfeval_base_moi": ("09_vidu_dong",     "self-eval trên cấu hình tốt nhất"),
}

# Ngưỡng 95 % KHÔNG phải một con số cố định: theo McNemar, dưới giả thuyết không thì
# hiệu số có độ lệch chuẩn √(b+c) mẫu. Hai cấu hình xáo trộn nhiều mẫu thì cần chênh
# lệch lớn hơn mới kết luận được. Đo thật: chỉ đổi card GPU đã làm nấc 03_sft xê dịch
# 2,4 điểm — nên mọi "sàn nhiễu" cố định đều là phán trên nhiễu.
def _mcnemar(a, b, key="ea"):
    x, y = R[a], R[b]
    if [r["id"] for r in x] != [r["id"] for r in y]:
        return None
    _b = sum(1 for p, q in zip(x, y) if p[key] and not q[key])
    _c = sum(1 for p, q in zip(x, y) if q[key] and not p[key])
    n = len(x)
    return (_c - _b) / n, _b, _c, 1.96 * ((_b + _c) ** 0.5) / n

print(f"\n{'─'*104}\n  PHẦN TĂNG THÊM CỦA TỪNG KỸ THUẬT (mỗi nấc so với nấc nó xây lên)"
      f"\n{'─'*104}")
print(f"{'kỹ thuật':<34}{'so với':<22}{'ΔEA':>9}{'hỏng':>6}{'sửa':>5}"
      f"{'ngưỡng':>9}   kết luận")
for _con, (_cha, _ten) in CHA.items():
    if _con not in R or _cha not in R:
        continue
    _r = _mcnemar(_cha, _con)
    if _r is None:
        print(f"{_ten:<34}{_cha[:21]:<22}{'—':>9}   hai nấc không cùng thứ tự mẫu")
        continue
    _d, _bb, _cc, _ng = _r
    _kl = "✅ tách khỏi nhiễu" if abs(_d) > _ng else "⚠ KHÔNG tách được"
    print(f"{_ten:<34}{_cha[:21]:<22}{_d*100:>+9.2f}{_bb:>6}{_cc:>5}"
          f"{_ng*100:>9.2f}   {_kl}")
print("  Δ tính bằng ĐIỂM EA (0–100). 'hỏng'/'sửa' là số mẫu chỉ một bên làm đúng.")
print("  Ngưỡng = 1,96·√(hỏng+sửa)/n — càng xáo trộn nhiều mẫu thì càng cần chênh lớn.")

# ── Ablation: tách riêng, so với đúng nấc gốc của nó ──
_abl = [("01_basic", "05c_ace_basic_base", "playbook ACE học trên prompt cơ bản"),
        ("04_selfeval_base", "04_selfeval_base_moi",
         "ô mục tiêu E+S: + K mẫu, ví dụ truy hồi, sửa-khi-lỗi"),
        ("05_ace_random_base", "05_ace_base",
         "truy hồi đúng bullet (vs ngẫu nhiên), prompt engineered"),
        ("05c_ace_basic_random_base", "05c_ace_basic_base",
         "truy hồi đúng bullet (vs ngẫu nhiên), prompt cơ bản")]
_abl = [(a, b, t) for a, b, t in _abl if a in R and b in R]
if _abl:
    print(f"\n{'─'*104}\n  ABLATION — nhánh rẽ, không nằm trên thang bậc\n{'─'*104}")
    print(f"{'thành phần được đo':<44}{'thiếu nó':>11}{'có nó':>10}{'đóng góp':>12}")
    for a, b, ten in _abl:
        print(f"{ten:<44}{M[a]['EA']:>11.4f}{M[b]['EA']:>10.4f}"
              f"{M[b]['EA']-M[a]['EA']:>+12.4f}")

## §4. Kiểm định từng bước leo thang

Các nấc chạy trên **cùng 497 mẫu** nên đây là dữ liệu *cặp*, và kiểm định đúng là
**McNemar**: chỉ nhìn các mẫu hai nấc bất đồng, đếm `b` (chỉ nấc trước đúng) và `c` (chỉ nấc
sau đúng). Kèm khoảng tin cậy 95 % bootstrap lấy mẫu lại theo cặp.

In [ ]:
print(f"\n{'═'*90}\n  KIỂM ĐỊNH (McNemar theo cặp, n={len(test_all)})\n{'═'*90}")
comparisons = []
# Dùng ĐÚNG cây CHA của ô trên, không phải `zip(_main, _main[1:])`. Chuỗi liền kề sai ở
# mọi chỗ rẽ nhánh — "Nấc 4 so với Nấc 3" không đo việc thêm kỹ thuật nào vì nấc 4 KHÔNG
# chạy trên adapter. Bảng in đã sửa từ trước; file `kiem_dinh_*.csv` mới là thứ đem đi
# báo cáo nên nó phải khớp, không được lệch.
_pairs = [(cha, con) for con, (cha, _t) in CHA.items() if con in R and cha in R]

# Đối chứng bullet NGẪU NHIÊN tách được "nội dung playbook" khỏi "cơ chế truy hồi":
# bullet ngẫu nhiên lấy từ CHÍNH playbook đã học, nên chênh lệch còn lại là do truy hồi.
for _ran, _ace in (("05_ace_random_base", "05_ace_base"),
                   ("05c_ace_basic_random_base", "05c_ace_basic_base")):
    if _ace in R and _ran in R:
        _pairs.append((_ran, _ace))
# ACE tự khám phá lại được bao nhiêu phần prompt viết tay
if "05c_ace_basic_base" in R and "01_basic" in R:
    _pairs.append(("01_basic", "05c_ace_basic_base"))
if "05c_ace_basic_random_base" in R and "01_basic" in R:
    _pairs.append(("01_basic", "05c_ace_basic_random_base"))

for a, b in _pairs:
    for key in ("ea", "pa_strict"):
        comparisons.append(stats.compare_pair(
            R[a], R[b], key=key,
            label=f"{M[b]['label']}  so với  {M[a]['label']}",
            name_base=a, name_variant=b))

_sig = [c for c in comparisons if c["p_value"] < 0.05 and c["key"] == "ea"]
print(f"\n  {len(_sig)}/{sum(1 for c in comparisons if c['key']=='ea')} bước leo thang "
      f"đạt p < 0.05 trên EA.")
print(f"  Với n={len(test_all)}, chênh lệch dưới ~2 điểm EA thường chưa đạt mức đó —")
print(f"  đó là giới hạn cỡ mẫu, không phải kỹ thuật thất bại.")

## §5. Kỹ thuật nào giúp loại bài nào

In [ ]:
_by_id = {s["id"]: s for s in test_all}

print(f"\n{'═'*92}\n  EA THEO ĐỘ PHỨC TẠP (số phép toán của gold)\n{'═'*92}")
# Lấy từ HỢP của mọi nấc, không phải một nấc bất kỳ: nấc đầu tiên có thể là bản
# chạy dở, khi đó cột độ phức tạp sẽ thiếu mà không ai biết.
_ks = sorted({r["n_ops_gold"] for rows in R.values() for r in rows
              if r["n_ops_gold"] > 0})
print(f"{'nấc':<34}" + "".join(f"{str(k) + ' phép':>10}" for k in _ks))
for s in _main:
    line = f"{M[s]['label']:<34}"
    for k in _ks:
        sub = [r for r in R[s] if r["n_ops_gold"] == k]
        line += f"{sum(r['ea'] for r in sub)/len(sub):>10.1%}" if sub else f"{'—':>10}"
    print(line)

print(f"\n{'═'*92}\n  EA THEO NGUỒN DỮ LIỆU\n{'═'*92}")
print(f"{'nấc':<34}{'FinQA-Vi':>12}{'Vi Data':>12}{'chênh':>10}")
for s in _main:
    g = {}
    for src in ("FinQA-Vi", "ViData"):
        sub = [r for r in R[s] if data.source_of(_by_id[r["id"]]) == src]
        g[src] = sum(r["ea"] for r in sub) / len(sub) if sub else 0
    print(f"{M[s]['label']:<34}{g['FinQA-Vi']:>12.1%}{g['ViData']:>12.1%}"
          f"{g['ViData']-g['FinQA-Vi']:>+10.1%}")

print(f"\n  Vi Data dùng table_* dày hơn hẳn FinQA-Vi; chênh lệch giữa hai cột cho biết")
print(f"  kỹ thuật nào giúp được nhóm câu đọc bảng.")

## §5b. EA/PA theo LOẠI phép toán — chỗ `table_*` hiện hình

`by_steps` gộp theo SỐ phép nên câu `table_*` tan vào ô "1 phép" và biến mất. Bảng dưới
tách theo LOẠI phép của gold, nên dòng `table_*` đọc thẳng được.

In [ ]:
# ─── EA/PA theo LOẠI phép toán của gold ───
# 59/61 mẫu table_* của tập test nằm trong ô "1 phép" (18,5 % ô đó). Gộp theo số phép
# thì không thấy; tách theo loại phép thì thấy ngay có đáng sửa prompt hay không.
_loai = []
for s in _main:
    for k in M[s].get("by_phep", {}):
        if k not in _loai:
            _loai.append(k)
_loai = sorted(_loai, key=lambda k: -max(M[s].get("by_phep", {}).get(k, [0])[0]
                                         for s in _main))
if _loai:
    print(f"\n{'═'*100}\n  EA THEO LOẠI PHÉP TOÁN (gold)\n{'═'*100}")
    print(f"{'nấc':<34}" + "".join(f"{k[:12]:>13}" for k in _loai))
    for s in _main:
        d = M[s].get("by_phep", {})
        line = f"{M[s]['label'][:33]:<34}"
        for k in _loai:
            tot, ea_, _pa = d.get(k, [0, 0, 0])
            line += f"{ea_/tot:>13.1%}" if tot else f"{'—':>13}"
        print(line)
    print(f"\n{'mẫu':<34}" + "".join(
        f"{max(M[s].get('by_phep', {}).get(k, [0])[0] for s in _main):>13}"
        for k in _loai))

    print(f"\n{'═'*100}\n  PA_strict THEO LOẠI PHÉP TOÁN (gold)\n{'═'*100}")
    print(f"{'nấc':<34}" + "".join(f"{k[:12]:>13}" for k in _loai))
    for s in _main:
        d = M[s].get("by_phep", {})
        line = f"{M[s]['label'][:33]:<34}"
        for k in _loai:
            tot, _ea, pa_ = d.get(k, [0, 0, 0])
            line += f"{pa_/tot:>13.1%}" if tot else f"{'—':>13}"
        print(line)
    print("\n  Dòng `table_*` = câu phải ĐỌC NHÃN HÀNG trong bảng. Thấp hơn hẳn các cột")
    print("  khác thì chỗ hụt nằm ở nhóm câu ĐỌC BẢNG, không phải ở self-eval hay ACE.")

# ─── Vì sao "không sinh được program": bị cắt vì trần hay sai định dạng ───
# Hai thứ này cần thuốc khác hẳn nhau. Gộp chung thì không biết đằng nào mà lần.
# Lấy từ META, KHÔNG từ M[s]. `summarize` cần `raw_step1` mới tính được khối này, mà
# jsonl cố tình bỏ output thô đi cho nhẹ — nên tính lại ở đây luôn ra None và bảng này
# không bao giờ in được gì. Meta thì đã lưu sẵn metrics tính lúc chạy, khi còn raw.
_vs = [(s, ((META.get(s) or {}).get("metrics") or {}).get("vi_sao_khong_co_program")
            or M[s].get("vi_sao_khong_co_program")) for s in _main]
_vs = [(s, v) for s, v in _vs if v]
if not _vs and _main:
    print("\n  ⊘ Không nấc nào có 'vi_sao_khong_co_program' trong meta — nấc "
          "chạy trước khi có trường này thì bỏ qua bảng dưới.")
if _vs:
    print(f"\n{'═'*100}\n  VÌ SAO KHÔNG SINH ĐƯỢC PROGRAM\n{'═'*100}")
    print(f"{'nấc':<34}{'bị cắt':>10}{'sai định dạng':>16}{'lặp (trung vị)':>17}"
          f"{'ca lặp nặng':>14}")
    for s, v in _vs:
        _l = v.get("lap_trung_vi")
        print(f"{M[s]['label'][:33]:<34}{v['bi_cat_giua_suy_nghi']:>10}"
              f"{v['sai_dinh_dang']:>16}"
              f"{(f'{_l:.0%}' if _l is not None else '—'):>17}"
              f"{v.get('so_ca_lap_nang', '—'):>14}")
    print("\n  Lặp trung vị CAO → quay vòng, nâng trần vô ích (thuốc ở repetition_penalty).")
    print("  Lặp THẤP mà vẫn bị cắt → suy luận dài thật; đã đo: nâng 4096→8192 cứu 0 mẫu,")
    print("  nên cách chữa là lượt VỚT với suy nghĩ tắt, không phải nâng trần.")


## §5d. Phương pháp mới — bóc tách từ dữ liệu thô, KHÔNG chạy lại GPU

Hai nấc `08`/`09` lưu **mọi mẫu đã sinh** (`cac_program`, `cac_gia_tri`, `cac_ea`,
`cac_pa`) và cả chương trình **trước** lượt sửa. Nhờ vậy từ hai lượt chạy GPU, mọi phép
so dưới đây tính được hết trên CPU.

In [ ]:
# ═══ Phương pháp mới: self-consistency · ví dụ động · lượt sửa · trần best-of-K ═══
_MOI = [s for s in ("08_tu_nhat_quan", "09_vidu_dong") if s in R]
if not _MOI:
    print("  ⊘ chưa có nấc phương pháp mới — chạy 08_phuong_phap_moi.ipynb")
else:
    _K = max(max((r.get("k_da_sinh") or 1) for r in R[s]) for s in _MOI)

    # ── 1. Đường cong self-consistency ──
    print(f"\n{'═'*96}\n  SELF-CONSISTENCY THEO k\n{'═'*96}")
    print(f"{'nấc':<26}" + "".join(f"{'k='+str(k):>11}" for k in range(1, _K+1))
          + f"{'trần best-of-K':>17}")
    _duong = {}
    for s in _MOI:
        _duong[s] = [pipeline.summarize(pipeline.tu_nhat_quan(R[s], test_all, k), s)
                     for k in range(1, _K+1)]
        _tr = pipeline.tran_best_of_k(R[s])
        print(f"{s:<26}" + "".join(f"{m['EA']:>11.4f}" for m in _duong[s])
              + f"{_tr['EA_tran']:>17.4f}")
    print("  (EA. Trần = luôn chọn được mẫu đúng nhất trong K — CẬN TRÊN, không đạt được.)")

    print(f"\n{'nấc':<26}" + "".join(f"{'k='+str(k):>11}" for k in range(1, _K+1))
          + f"{'trần best-of-K':>17}")
    for s in _MOI:
        _tr = pipeline.tran_best_of_k(R[s])
        print(f"{s:<26}" + "".join(f"{m['PA_strict']:>11.4f}" for m in _duong[s])
              + f"{_tr['PA_tran']:>17.4f}")
    print("  (PA_strict.)")

    # ── 2. Từng phương pháp đóng góp bao nhiêu, kèm KTC ──
    print(f"\n{'═'*96}\n  TÁCH ĐÓNG GÓP TỪNG PHƯƠNG PHÁP\n{'═'*96}")
    _viec = []
    for s in _MOI:
        if _K > 1:
            _viec.append((f"self-consistency k=1→{_K} [{s}]",
                          pipeline.tu_nhat_quan(R[s], test_all, 1),
                          pipeline.tu_nhat_quan(R[s], test_all, _K)))
    if len(_MOI) == 2:
        _a, _b = "08_tu_nhat_quan", "09_vidu_dong"
        _viec.append(("ví dụ truy hồi (k=1)",
                      pipeline.tu_nhat_quan(R[_a], test_all, 1),
                      pipeline.tu_nhat_quan(R[_b], test_all, 1)))
        _viec.append((f"ví dụ truy hồi (k={_K})",
                      pipeline.tu_nhat_quan(R[_a], test_all, _K),
                      pipeline.tu_nhat_quan(R[_b], test_all, _K)))
    if "02_prompt_eng" in R:
        _viec.append(("nhiệt độ 0.1 → cao (k=1, cùng prompt)",
                      R["02_prompt_eng"],
                      pipeline.tu_nhat_quan(R["08_tu_nhat_quan"], test_all, 1)
                      if "08_tu_nhat_quan" in R else None))
    for _ten, _x, _y in _viec:
        if _x is None or _y is None:
            continue
        for _key in ("ea", "pa_strict"):
            comparisons.append(stats.compare_pair(
                _x, _y, key=_key, label=_ten, name_base="trước", name_variant="sau"))

    # ── 3. Lượt sửa đóng góp bao nhiêu ──
    print(f"\n{'═'*96}\n  LƯỢT SỬA (program bị executor từ chối → sinh lại kèm lý do)\n{'═'*96}")
    print(f"{'nấc':<26}{'số mẫu sửa':>13}{'EA có sửa':>12}{'EA nếu bỏ sửa':>16}{'Δ':>9}")
    for s in _MOI:
        _n = sum(1 for r in R[s] if r.get("da_sua"))
        if not _n:
            print(f"{s:<26}{0:>13}{'—':>12}{'—':>16}{'—':>9}")
            continue
        _bo = []
        for r in R[s]:                      # dựng lại bản KHÔNG sửa từ program_truoc_sua
            q = dict(r)
            if r.get("da_sua"):
                q["cac_program"] = [r.get("program_truoc_sua") or ""]
                q["cac_gia_tri"] = [None]
            _bo.append(q)
        # So ở ĐÚNG mức k của nấc, không phải k=1 — nếu không thì con số "Δ do lượt
        # sửa" nuốt luôn phần đóng góp của self-consistency.
        _kk = max((r.get("k_da_sinh") or 1) for r in R[s])
        _mb = pipeline.summarize(pipeline.tu_nhat_quan(_bo, test_all, _kk), s)
        _mc = pipeline.summarize(pipeline.tu_nhat_quan(R[s], test_all, _kk), s)
        print(f"{s:<26}{_n:>13}{_mc['EA']:>12.4f}{_mb['EA']:>16.4f}"
              f"{_mc['EA']-_mb['EA']:>+9.4f}")

    # ── 4. Mức đồng thuận: bao nhiêu phiếu cho đáp án thắng ──
    print(f"\n{'═'*96}\n  ĐỒNG THUẬN GIỮA K MẪU — đồng thuận cao có đi kèm đúng không?\n{'═'*96}")
    print(f"{'nấc':<26}{'số phiếu':>10}{'số mẫu':>9}{'EA':>9}")
    for s in _MOI:
        _nhom = defaultdict(list)
        for r in R[s]:
            _nhom[r.get("so_phieu") or 0].append(r)
        for _p in sorted(_nhom, reverse=True):
            _g = _nhom[_p]
            print(f"{s if _p == max(_nhom) else '':<26}{_p:>10}{len(_g):>9}"
                  f"{sum(x['ea'] for x in _g)/len(_g):>9.1%}")
    print("\n  Đồng thuận cao mà EA cũng cao → số phiếu dùng được làm ĐỘ TIN CẬY.")
    print("  Đồng thuận cao mà EA thấp → model sai một cách nhất quán; bỏ phiếu không cứu được.")


## §5c. Cổng bước 2 — tính lại từ jsonl, KHÔNG tốn GPU

Mặc định self-eval lấy program bước 2 vô điều kiện, kể cả khi bước 2 sinh ra một
chương trình **không chạy được** còn bước 1 thì chạy được. Cổng chỉ hỏi executor, không
đụng đáp án vàng — và vì jsonl đã lưu cả `program_step1` lẫn `program_step2`, nấc "có
cổng" tính thẳng ra ở đây, không phải chạy lại model.

In [ ]:
# ─── Nấc self-eval CÓ CỔNG, dựng lại trên CPU ───
_co_se = [s for s in _main if any((r.get("program_step2") or "").strip() for r in R[s])]
if _co_se:
    print(f"\n{'═'*100}\n  CỔNG BƯỚC 2 (tính lại từ jsonl, không chạy model)\n{'═'*100}")
    print(f"{'nấc':<34}{'EA gốc':>9}{'EA có cổng':>12}{'PA gốc':>9}{'PA có cổng':>12}"
          f"{'mẫu được cứu':>14}")
    for s in _co_se:
        _g = pipeline.ap_cong_buoc2(R[s], test_all)
        _mg = pipeline.summarize(_g, M[s]["label"] + " + cổng")
        _cuu = sum(1 for a, b in zip(R[s], _g) if not a["ea"] and b["ea"])
        _mat = sum(1 for a, b in zip(R[s], _g) if a["ea"] and not b["ea"])
        print(f"{M[s]['label'][:33]:<34}{M[s]['EA']:>9.4f}{_mg['EA']:>12.4f}"
              f"{M[s]['PA_strict']:>9.4f}{_mg['PA_strict']:>12.4f}"
              f"{_cuu - _mat:>+14}")
        comparisons.append(stats.compare_pair(
            R[s], _g, key="ea", label=f"{M[s]['label']} — có cổng so với không cổng",
            name_base=s, name_variant=s + "_cong"))
    print("\n  Δ = 0 ở đây nghĩa là bước 2 chưa bao giờ sinh ra program hỏng trên mẫu mà")
    print("  bước 1 làm đúng — kết quả âm tính, ghi đúng như vậy, đừng tô thành gì khác.")


## §6. Chuyển dịch kiểu lỗi qua các nấc

In [ ]:
_outs = sorted({o for s in _main for o in M[s]["outcome"]})
print(f"\n{'═'*100}\n  PHÂN BỐ KẾT CỤC\n{'═'*100}")
print(f"{'nấc':<34}" + "".join(f"{o[:14]:>16}" for o in _outs))
for s in _main:
    print(f"{M[s]['label']:<34}" +
          "".join(f"{M[s]['outcome'].get(o, 0):>16}" for o in _outs))
print(f"\n  Hai cột đáng theo dõi nhất: 'khong_co_program' (model không sinh nổi khối")
print(f"  plaintext) và 'program_khong_chay_duoc' (sinh được nhưng executor từ chối).")
print(f"  Cả hai giảm dần là dấu hiệu các ràng buộc định dạng đang có tác dụng.")

# ─── Vì sao executor TỪ CHỐI, tách theo lý do ───
# "program không chạy được" từng là ô lớn nhất bảng trên (33 % ở nấc prompt cơ bản),
# lớn hơn cả ô "sai". Gộp chung thì không biết kỹ thuật nào chữa được lỗi nào; tách ra
# thì thấy ngay, ví dụ nếu `nhan_bang_khong_khop` tụt mạnh ở nấc 2 thì phần thắng của
# prompt engineering nằm ở chỗ dạy ĐỌC BẢNG, không phải ở quy tắc định dạng.
_ld = sorted({k for s in _main
              for k in ((M[s].get("vi_sao_khong_chay_duoc") or {}).get("theo_ly_do") or {})})
if _ld:
    print(f"\n{'─'*100}\n  VÌ SAO EXECUTOR TỪ CHỐI (số mẫu)\n{'─'*100}")
    print(f"{'nấc':<34}" + "".join(f"{k[:15]:>17}" for k in _ld))
    for s in _main:
        _t = (M[s].get("vi_sao_khong_chay_duoc") or {}).get("theo_ly_do") or {}
        print(f"{M[s]['label'][:33]:<34}" + "".join(f"{_t.get(k, 0):>17}" for k in _ld))
    print(f"\n  nhan_bang_khong_khop = gọi table_* với nhãn không hàng nào khớp"
          f" → cần dạy đọc bảng")
    print(f"  phep_long_nhau       = divide(5310, add(1, 0.15)) → prompt có dòng cấm,"
          f" đây là thước đo dòng đó")
    print(f"  tham_chieu_sai       = #N trỏ tới phép chưa có → lỗi lập kế hoạch nhiều bước")

# ─── Bước 2 (self-eval / ACE) đổi được bao nhiêu program ───
# Δ EA = 0 vẫn còn hai cách giải thích rất khác nhau: bước 2 chép lại y nguyên bước 1
# (không tìm ra gì để sửa), hay nó sửa nhiều mà toàn sửa hình thức. Đếm thẳng mới biết.
# Chạy được trên nấc ĐÃ XONG vì jsonl có sẵn program_step1 và program_step2.
_hb_co = [(s, pipeline.so_sanh_hai_buoc(R[s], test_all)) for s in _main if s in R]
_hb_co = [(s, h) for s, h in _hb_co if h]
if _hb_co:
    print(f"\n{'─'*100}\n  BƯỚC 2 ĐỔI ĐƯỢC BAO NHIÊU PROGRAM\n{'─'*100}")
    _cot = ["chep_lai_y_nguyen", "doi_nhung_gia_tri_giu_nguyen", "doi_va_doi_ca_gia_tri",
            "cuu_mau_buoc1_bo_trong", "buoc2_bo_trong_giu_buoc1"]
    print(f"{'nấc':<26}{'đổi':>7}" + "".join(f"{c[:15]:>17}" for c in _cot))
    for s, h in _hb_co:
        print(f"{s:<26}{h['ty_le_bi_doi']:>7.1%}"
              + "".join(f"{h['theo_nhom'].get(c, 0):>17}" for c in _cot))
    for s, h in _hb_co:
        _yn = h["theo_nhom"].get("chep_lai_y_nguyen", 0) / max(1, len(R[s]))
        if _yn > 0.9:
            print(f"\n  ⚠ {s}: bước 2 chép lại y nguyên {_yn:.0%} số mẫu — nó không tìm")
            print("     ra gì để sửa. Đó là vì bước 1 hầu như không còn lỗi ĐỊNH DẠNG,")
            print("     mà lỗi còn lại là suy luận sai — thứ một lượt tự soát khó bắt.")

# ─── Sai KIỂU gì ───
# Sau nấc 2 thì "sai" là ô LỚN NHẤT (136/497 = 27 %). Tách theo kiểu mới biết chữa gì:
# nhầm phép → sửa ánh xạ từ khoá; nhầm SỐ mà dãy phép trùng khít gold → dạy đọc bảng;
# thiếu bước → dạy lập kế hoạch nhiều bước. Ba thứ cần ba câu khác nhau trong prompt.
_ks = sorted({k for s in _main for k in ((M[s].get("vi_sao_sai") or {}).get("theo_kieu") or {})})
if _ks:
    print(f"\n{'─'*100}\n  SAI KIỂU GÌ (số mẫu)\n{'─'*100}")
    print(f"{'nấc':<30}" + "".join(f"{k[:16]:>18}" for k in _ks))
    for s in _main:
        _t = (M[s].get("vi_sao_sai") or {}).get("theo_kieu") or {}
        print(f"{M[s]['label'][:29]:<30}" + "".join(f"{_t.get(k, 0):>18}" for k in _ks))
    _cuoi = _main[-1]
    _vd = (M[_cuoi].get("vi_sao_sai") or {}).get("vi_du") or {}
    if _vd:
        print(f"\n  Ví dụ thật ở nấc cuối ({_cuoi}):")
        for _k, _ds in list(_vd.items())[:3]:
            for _e in _ds[:1]:
                print(f"    [{_k}] {_e['hoi']}")
                print(f"        model = {_e['model']}")
                print(f"        gold  = {_e['gold']}")


## §7. Đối chiếu mốc tham chiếu

In [ ]:
_ref = io_utils.BASELINE_RESULTS["Qwen3-8B"]
_pub_csv = os.path.join(REPO_DIR, "reference", "baseline_results", "Qwen3-8B_program.csv")

print(f"{'═'*88}\n  SO VỚI MỐC THAM CHIẾU\n{'═'*88}")
print(f"  Tham chiếu, Qwen3-8B + self-eval : PA = {_ref['PA']:.2f}%   EA = {_ref['EA']:.2f}%")
if os.path.exists(_pub_csv):
    _pr = io_utils.score_saved_predictions(_pub_csv, test_all, "tham chiếu")
    _pm = pipeline.summarize(_pr, "tham chiếu")
    print(f"  Chính dự đoán đó, chấm lại      : PA_loose = {_pm['PA_loose']*100:.2f}%   "
          f"EA = {_pm['EA']*100:.2f}%")
    print(f"    (EA cao hơn vì executor cũ bỏ sót một phần câu table_*)")

for s in _main:
    if s.startswith("04_selfeval") or s.startswith("05_ace"):
        print(f"  {M[s]['label']:<32}: PA_loose = {M[s]['PA_loose']*100:.2f}%   "
              f"EA = {M[s]['EA']*100:.2f}%")

print(f"\n  ⚠ Chỉ PA_loose so trực tiếp được với cột PA tham chiếu.")
print(f"    EA tham chiếu tính bằng executor cũ nên thấp hơn thực tế.")

## §8. Xuất bảng và biểu đồ

In [ ]:
_csv = os.path.join(OUTPUT_DIR, f"bang_ket_qua_{STAMP}.csv")
with open(_csv, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["nac", "mo_ta", "n", "EA", "PA_strict", "PA_loose",
                                      "no_program", "exec_none", "phut"])
    w.writeheader()
    for s in _main:
        m = M[s]
        w.writerow({"nac": s, "mo_ta": m["label"], "n": m["n"],
                    "EA": round(m["EA"]*100, 2), "PA_strict": round(m["PA_strict"]*100, 2),
                    "PA_loose": round(m["PA_loose"]*100, 2),
                    "no_program": round(m["no_program"]*100, 2),
                    "exec_none": round(m["exec_none"]*100, 2),
                    "phut": META.get(s, {}).get("metrics", {}).get("minutes", "")})

_cmp = os.path.join(OUTPUT_DIR, f"kiem_dinh_{STAMP}.csv")
with open(_cmp, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["so_sanh", "chi_so", "base", "variant", "delta",
                                      "ci_low", "ci_high", "p_value", "b", "c", "ket_luan"])
    w.writeheader()
    for c in comparisons:
        w.writerow({"so_sanh": c["label"], "chi_so": c["key"], "base": c["base"],
                    "variant": c["variant"], "delta": c["delta"],
                    "ci_low": c["ci95"][0], "ci_high": c["ci95"][1],
                    "p_value": round(c["p_value"], 5), "b": c["b"], "c": c["c"],
                    "ket_luan": c["verdict"]})
print(f"[SAVE] {_csv}\n[SAVE] {_cmp}")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

_short = [M[s]["label"].split("—")[0].strip() for s in _main]
x = np.arange(len(_main))
for i, (k, lbl) in enumerate([("EA", "EA"), ("PA_strict", "PA")]):
    vals = [M[s][k] for s in _main]
    bars = axes[0].bar(x + i*0.38, vals, 0.38, label=lbl)
    for b, v in zip(bars, vals):
        axes[0].text(b.get_x()+b.get_width()/2, v+0.008, f"{v:.3f}", ha="center", fontsize=7)
axes[0].set_xticks(x + 0.19); axes[0].set_xticklabels(_short, rotation=20, ha="right", fontsize=8)
axes[0].set_ylim(0, 1); axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)
axes[0].set_title(f"Thang bậc — ViNumQA test ({len(test_all)} mẫu)")

_ea = [c for c in comparisons if c["key"] == "ea" and not c["base"].endswith("random_base")]
if _ea:
    lab = [f"{c['variant'][:14]}\nvs {c['base'][:14]}" for c in _ea]
    dl = [c["delta"] for c in _ea]
    er = [[c["delta"]-c["ci95"][0] for c in _ea], [c["ci95"][1]-c["delta"] for c in _ea]]
    col = ["tab:green" if c["p_value"] < 0.05 and c["delta"] > 0 else
           "tab:red" if c["p_value"] < 0.05 else "tab:gray" for c in _ea]
    axes[1].bar(lab, dl, color=col, yerr=er, capsize=4)
    for i, c in enumerate(_ea):
        axes[1].text(i, c["delta"], f"p={c['p_value']:.3f}", ha="center",
                     va="bottom" if c["delta"] >= 0 else "top", fontsize=7)
axes[1].axhline(0, color="black", lw=0.8); axes[1].set_ylabel("ΔEA (thanh = KTC 95%)")
axes[1].tick_params(axis="x", labelsize=7); axes[1].grid(axis="y", alpha=0.3)
axes[1].set_title("Phần tăng thêm của từng kỹ thuật")

for s in _main:
    bs = M[s]["by_steps"]
    ks = sorted(bs, key=int)
    axes[2].plot(ks, [bs[k][1]/bs[k][0] for k in ks], marker="o",
                 label=M[s]["label"].split("—")[0].strip())
axes[2].set_xlabel("số phép toán trong gold"); axes[2].set_ylabel("EA")
axes[2].set_ylim(0, 1); axes[2].legend(fontsize=7); axes[2].grid(alpha=0.3)
axes[2].set_title("EA theo độ phức tạp")

plt.tight_layout()
_png = os.path.join(OUTPUT_DIR, f"bao_cao_{STAMP}.png")
plt.savefig(_png, dpi=150); plt.show()
print(f"[SAVE] {_png}")

## §9. Đọc kết quả cho đúng

| Điều cần nhớ | Vì sao |
|---|---|
| Chỉ **`PA_loose`** so trực tiếp được với bảng tham chiếu | `PA_strict` chặt hơn; EA cũ tính bằng executor lỗi `table_*` |
| Chênh lệch < ~2 điểm EA thường **không** đạt p < 0.05 | n = 497 là nhỏ. Muốn chắc hơn: chạy thêm trên valid (584 mẫu) rồi gộp |
| Không trộn kết quả giữa máy có cắt ngữ cảnh và máy không cắt | mỗi notebook in rõ ở dòng `[PROMPT]` |
| Chạy nhiều cấu hình thì dễ có cái "đạt p<0.05" do may mắn | kết luận mạnh chỉ nên dựa vào các bước leo thang đã định trước |
| Nấc 4 và nấc 5 đều là cơ chế sửa lỗi lúc suy luận | nếu ACE ≈ self-eval thì nhiều khả năng **chồng lấn**, không phải ACE vô dụng |

## Cái giá của từng kỹ thuật

Ngoài EA/PA, dự án đặt trong bối cảnh **tài nguyên hạn chế**, nên nên báo cáo kèm:

| Nấc | Lượt sinh / mẫu | Cần huấn luyện? | Cần API ngoài? |
|---|:---:|:---:|:---:|
| 1, 2 | 1 | ❌ | ❌ |
| 3 (SFT) | 1 | ✅ một lần | ❌ |
| 4 (self-eval) | 2 | ❌ | ❌ |
| 5 (ACE) | 2 + pha A một lần | ❌ | ❌ |

Cột cuối là điểm mạnh chung của cả lộ trình: **không nấc nào cần API ngoài**, nên toàn bộ
nằm trong thiết lập *constrained-resource* của dự án — khác với nhánh Phi-4 + Gemini
(unconstrained) vốn còn cho kết quả thấp hơn.